# Problem Set 2 — Question 2: Autoencoder Algorithms
**Course:** 1.121 / 2.174 / 1.052J — Advancing Mechanics and Materials via Machine Learning  
**Due:** March 22, 2026

---

**Goal:** Develop a convolutional autoencoder trained on 1,000 GRF two-phase microstructures (PNG + NPZ subset from the MICRO2D dataset). Use it to reproduce training data and generate novel synthetic microstructures.

**Dataset:** 1,000 GRF microstructures provided via Dropbox  
**Reference:** Robertson et al., *Integrating Materials and Manufacturing Innovation*, 2023. https://doi.org/10.1007/s40192-023-00340-4

## Imports & Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch version: {torch.__version__}")

# Device selection
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

---
## Part (a) — Dataset Analysis and Sample Images

In [ ]:
NPZ_PATH = "grf_data/MICRO2D_GRF_1k.npz"

data = np.load(NPZ_PATH, allow_pickle=True)
images_raw = data['X']

print("=== Dataset Summary ===")
print(f"Number of samples : {images_raw.shape[0]}")
print(f"Original image size: {images_raw.shape[1]} x {images_raw.shape[2]} pixels")
print(f"Data type          : {images_raw.dtype}")
print(f"Pixel value range  : [{images_raw.min():.3f}, {images_raw.max():.3f}]")
print(f"Mean pixel value   : {images_raw.mean():.3f}")
print(f"Keys in .npz file  : {list(data.keys())}")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("Part (a): Sample GRF Microstructures", fontsize=14)

for idx, ax in enumerate(axes.flat):
    ax.imshow(images_raw[idx], cmap='gray')
    ax.set_title(f"Sample {idx+1}")
    ax.axis('off')

plt.tight_layout()
plt.savefig("part_a_sample_images.png", dpi=150)
plt.show()

**Observations:**  
The dataset contains 1,000 binary (black-and-white) two-phase microstructures generated using a Gaussian Random Field (GRF) model. Each image represents a cross-section of a two-phase material where one phase is white and the other is black. The microstructures exhibit a wide range of spatial patterns:
- **Isotropic/blob-like** structures (Samples 1, 4, 5, 8): irregular phase boundaries with no preferred direction.
- **Anisotropic/lamellar** structures (Samples 3, 9): high-frequency stripe patterns indicating strong directional correlation.
- **Sparse/elongated** structures (Samples 2, 6, 7): one phase forms elongated islands within a matrix of the other.

This diversity results from varying the GRF correlation length and threshold. The dataset is well-balanced in phase fraction (mean ≈ 0.5), which is typical of GRF-generated data.

---
## Part (b) — Image Resolution Choice

**Chosen resolution: 64 × 64 pixels**

We tested two resolutions:

| Resolution | Result |
|------------|--------|
| **64 × 64** | **Successful — selected** |
| 128 × 128  | Failed — VAE posterior collapse (all-black outputs) |

**What happened at 128×128:** The model ran without crashing, but all reconstructions and generated images were completely black. This is a **VAE posterior collapse**: the KL divergence loss overwhelmed the reconstruction loss early in training, causing the decoder to learn all-zero outputs as a trivial minimum. The larger spatial resolution (128×128) combined with a heavier architecture (512 channels, latent dim 128) made the model more prone to this instability.

**Why 64×64 works:** The smaller architecture is easier to train stably with BCE + KL losses. All key GRF morphological features — blob boundaries, stripe periodicity, sparse inclusions — are fully captured at 64×64 for binary images. The encoder compresses: 64→32→16→8→4 over 4 strided-conv layers, producing a 4×4 feature map before the latent bottleneck.

---
## Part (c) — Convolutional Autoencoder: Architecture, Training, and Performance

In [ ]:
def load_and_prepare_data(npz_path, target_size=64):
    """Load, resize, and binarize the GRF dataset."""
    data = np.load(npz_path, allow_pickle=True)
    images = data['X']
    resized = []
    for img in images:
        img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)
        pil_img = Image.fromarray(img_uint8).resize((target_size, target_size), Image.LANCZOS)
        resized.append(np.array(pil_img))
    images_norm = np.array(resized).astype(np.float32) / 255.0
    images_norm = (images_norm > 0.5).astype(np.float32)  # binarize
    return images_norm


class ConvAutoencoder(nn.Module):
    """
    Convolutional Autoencoder for binary GRF microstructures.

    Architecture:
      Encoder: 4 × Conv2d (stride=2) with BatchNorm + LeakyReLU → flatten → FC → latent (dim=64)
      Decoder: FC → reshape → 4 × ConvTranspose2d with BatchNorm + LeakyReLU → Sigmoid output

    Input/Output: 1 × 64 × 64 binary image
    Latent dim  : 64
    """
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim

        # --- Encoder ---
        # Each Conv2d: kernel=4, stride=2, padding=1 → halves spatial dims
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1),   nn.BatchNorm2d(32),  nn.LeakyReLU(0.2),  # 64→32
            nn.Conv2d(32, 64, 4, 2, 1),  nn.BatchNorm2d(64),  nn.LeakyReLU(0.2),  # 32→16
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),  # 16→8
            nn.Conv2d(128, 256, 4, 2, 1),nn.BatchNorm2d(256), nn.LeakyReLU(0.2),  # 8→4
        )
        self.fc_enc = nn.Linear(256 * 4 * 4, latent_dim)

        # --- Decoder ---
        self.fc_dec = nn.Linear(latent_dim, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),  # 4→8
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  nn.BatchNorm2d(64),  nn.LeakyReLU(0.2),  # 8→16
            nn.ConvTranspose2d(64, 32, 4, 2, 1),   nn.BatchNorm2d(32),  nn.LeakyReLU(0.2),  # 16→32
            nn.ConvTranspose2d(32, 1, 4, 2, 1),    nn.Sigmoid(),                            # 32→64
        )

    def encode(self, x):
        return self.fc_enc(self.encoder(x).view(x.size(0), -1))

    def decode(self, z):
        return self.decoder(self.fc_dec(z).view(z.size(0), 256, 4, 4))

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


# Print architecture summary
model = ConvAutoencoder(latent_dim=64)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

**Architecture choices:**

| Component | Choice | Reason |
|-----------|--------|--------|
| Conv layers | 4 strided convolutions | Each halves spatial dims; captures multi-scale features |
| Activation | LeakyReLU (α=0.2) | Avoids dead neurons common with ReLU in encoders |
| Normalization | BatchNorm after each conv | Stabilizes training, faster convergence |
| Latent dim | 64 | Compact enough to force compression; large enough to capture diversity |
| Output activation | Sigmoid | Maps output to [0,1] for BCE loss; appropriate for binary images |
| Loss function | Binary Cross-Entropy (BCE) | Correct choice for binary images — treats each pixel as a Bernoulli variable |

In [ ]:
def train_autoencoder(model, train_loader, num_epochs=150, lr=1e-3, device='cpu'):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    criterion = nn.BCELoss()
    losses = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for (batch_x,) in train_loader:
            batch_x = batch_x.to(device)
            optimizer.zero_grad()
            recon_x, _ = model(batch_x)
            loss = criterion(recon_x, batch_x)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        scheduler.step(avg_loss)
        if (epoch + 1) % 25 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]  Loss: {avg_loss:.6f}  LR: {optimizer.param_groups[0]['lr']:.6f}")

    return losses


# Load data and train
images_norm = load_and_prepare_data(NPZ_PATH, target_size=64)
train_tensor = torch.FloatTensor(images_norm).unsqueeze(1)
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=32, shuffle=True)

losses = train_autoencoder(model, train_loader, num_epochs=150, lr=1e-3, device=device)
torch.save(model.state_dict(), "autoencoder_model.pth")
print("Model saved.")

In [ ]:
# Training loss curve
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel("Epoch"); plt.ylabel("Loss (BCE)")
plt.title("Part (c): Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("part_c_training_loss.png", dpi=150)
plt.show()
print(f"Final training loss: {losses[-1]:.6f}")

In [ ]:
# Reconstructions
model.eval()
with torch.no_grad():
    test_tensor = torch.FloatTensor(images_norm[:5]).unsqueeze(1).to(device)
    recon, _ = model(test_tensor)
    recon = recon.cpu().numpy()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Part (c): Original vs Reconstructed", fontsize=14)
for i in range(5):
    axes[0, i].imshow(images_norm[i], cmap='gray'); axes[0, i].set_title(f"Original {i+1}"); axes[0, i].axis('off')
    recon_bin = (recon[i, 0] > 0.5).astype(np.float32)
    axes[1, i].imshow(recon_bin, cmap='gray'); axes[1, i].set_title(f"Reconstructed {i+1}"); axes[1, i].axis('off')
plt.tight_layout()
plt.savefig("part_c_reconstructions.png", dpi=150)
plt.show()

**Results:**  
The training loss (BCE) decreases steeply from ~0.54 at epoch 1 to ~0.02 at epoch 150, indicating successful learning. The `ReduceLROnPlateau` scheduler further refined convergence by halving the learning rate whenever progress stalled.

The reconstruction quality is high: all major morphological features are preserved — blob boundaries, stripe periodicity and orientation, and sparse inclusion layouts. Minor pixel-level deviations are present in the fine stripe patterns (Sample 3) due to the high-frequency content being harder to compress into 64 latent dimensions, but the overall structure is faithfully recovered.

---
## Part (d) — Latent Space Exploration

In [ ]:
# Extract all latent codes
model.eval()
with torch.no_grad():
    all_tensor = torch.FloatTensor(images_norm).unsqueeze(1).to(device)
    latent_codes = model.encode(all_tensor).cpu().numpy()

print(f"Latent codes shape: {latent_codes.shape}")
print(f"Mean: {latent_codes.mean():.3f}  Std: {latent_codes.std():.3f}")
print(f"Min: {latent_codes.min():.3f}  Max: {latent_codes.max():.3f}")

In [ ]:
# Visualize 2D projections of latent space — multiple dimension pairs
dim_pairs = [(0, 1), (2, 3), (4, 5), (10, 11), (20, 21), (30, 31)]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Part (d): Latent Space — 2D Projections (Different Dimension Pairs)", fontsize=14)

for ax, (d1, d2) in zip(axes.flat, dim_pairs):
    ax.scatter(latent_codes[:, d1], latent_codes[:, d2], alpha=0.4, s=10)
    ax.set_xlabel(f"Dim {d1}"); ax.set_ylabel(f"Dim {d2}")
    ax.set_title(f"Dim {d1} vs Dim {d2}")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("part_d_latent_space.png", dpi=150)
plt.show()

# Compute pairwise correlations for first 10 dims
corr_matrix = np.corrcoef(latent_codes[:, :10].T)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title("Part (d): Correlation Matrix — First 10 Latent Dimensions")
ax.set_xlabel("Latent dimension"); ax.set_ylabel("Latent dimension")
plt.tight_layout()
plt.savefig("part_d_correlation_matrix.png", dpi=150)
plt.show()

**Observations:**

1. **No strong linear correlations** between dimensions: the scatter plots show approximately circular, uncorrelated distributions for all dimension pairs tested (dims 0–1, 2–3, 4–5, 10–11, 20–21, 30–31). Points are spread in roughly isotropic Gaussian clouds.

2. **The correlation matrix** for the first 10 dimensions shows off-diagonal values close to 0, confirming that the encoder has learned approximately independent representations. This is characteristic of a standard (non-variational) autoencoder — it learns a deterministic mapping without any explicit regularization enforcing independence.

3. **Implication for generation (Part e):** Because the latent space has no explicit structure (unlike a VAE where latent codes are forced to follow N(0,I)), random sampling from outside the range of training codes may produce unrealistic outputs. Sampling near the empirical mean ± std of each dimension is more reliable.

---
## Part (e) — Generating New Microstructures from Custom Latent Codes

In [ ]:
# Strategy 1: Random codes sampled from learned latent distribution (mean ± 0.8*std)
np.random.seed(42)
mean = latent_codes.mean(axis=0)
std  = latent_codes.std(axis=0)

random_codes = np.random.randn(9, latent_codes.shape[1]) * std * 0.8 + mean

model.eval()
with torch.no_grad():
    z = torch.FloatTensor(random_codes).to(device)
    generated = model.decode(z).cpu().numpy()

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("Part (e): Generated Microstructures (Random Latent Codes, scaled by learned std)", fontsize=13)
for idx, ax in enumerate(axes.flat):
    gen_bin = (generated[idx, 0] > 0.5).astype(np.float32)
    ax.imshow(gen_bin, cmap='gray'); ax.set_title(f"Generated {idx+1}"); ax.axis('off')
plt.tight_layout()
plt.savefig("part_e_generated.png", dpi=150)
plt.show()

In [ ]:
# Strategy 2: Interpolation between two real microstructures in latent space
model.eval()
with torch.no_grad():
    img1 = torch.FloatTensor(images_norm[0:1]).unsqueeze(1).to(device)
    img2 = torch.FloatTensor(images_norm[50:51]).unsqueeze(1).to(device)
    z1 = model.encode(img1)
    z2 = model.encode(img2)

    alphas = np.linspace(0, 1, 7)
    interp_imgs = []
    for alpha in alphas:
        z_interp = (1 - alpha) * z1 + alpha * z2
        interp_imgs.append(model.decode(z_interp).cpu().numpy()[0, 0])

fig, axes = plt.subplots(1, 7, figsize=(14, 2.5))
fig.suptitle("Part (e): Latent Space Interpolation (α=0 → 1)", fontsize=13)
for idx, (ax, alpha) in enumerate(zip(axes, alphas)):
    ax.imshow((interp_imgs[idx] > 0.5).astype(float), cmap='gray')
    ax.set_title(f"α={alpha:.2f}"); ax.axis('off')
plt.tight_layout()
plt.savefig("part_e_interpolation.png", dpi=150)
plt.show()

**Discussion:**

**Can random latent codes generate "real" GRF-type microstructures?**  
*Partially.* The generated images (Strategy 1) exhibit convincing two-phase blob and irregular boundary textures that visually resemble GRF microstructures. However, some samples show artifacts — scattered isolated pixels or unrealistically fragmented regions — especially when the sampled code falls in a region of latent space that was not well-covered by training data.

This connects directly to **Part (d)**: because the standard autoencoder does not regularize the latent space, there are gaps and unexplored regions. Codes sampled from these gaps decode to unrealistic images. Sampling within ±0.8σ of the empirical mean (as done here) keeps the codes near the training manifold and improves quality.

**Interpolation (Strategy 2)** demonstrates that the latent space is continuous and semantically meaningful: the transition from one morphology to another is smooth, passing through plausible intermediate microstructures. This is strong evidence that the encoder has learned a useful latent representation.

---
## Part (f) — Approaches Beyond CNN Autoencoders for Microstructure Generation

The standard convolutional autoencoder reconstructs well but has a fragmented, unstructured latent space making controlled generation difficult. The following approaches address this limitation:

---

### 1. Variational Autoencoder (VAE)
**How it works:** The encoder outputs parameters (μ, σ) of a Gaussian distribution instead of a single point. The latent code is sampled as z = μ + ε·σ (reparameterization trick). A KL-divergence regularization term forces the latent distribution toward N(0, I).  
**Advantage over standard AE:** The latent space is smooth and well-structured — any point sampled from N(0, I) decodes to a plausible microstructure. Interpolation is more reliable.  
**Limitation:** The KL term trades reconstruction sharpness for smoothness; outputs can appear slightly blurry.

---

### 2. Generative Adversarial Network (GAN)
**How it works:** A generator G maps random noise z~N(0,I) to images; a discriminator D tries to distinguish real from generated images. Trained adversarially.  
**Advantage:** Produces sharper, more realistic images than VAE — excellent for binary/texture microstructures.  
**Limitation:** Training instability (mode collapse), difficult to control generated properties, no direct encoder for latent code manipulation.

---

### 3. Denoising Diffusion Probabilistic Models (DDPM)
**How it works:** Learns to reverse a gradual Gaussian noise process. At generation time, pure noise is iteratively denoised into a realistic image.  
**Advantage:** State-of-the-art image quality and diversity; stable training; supports conditioning on material properties.  
**Limitation:** Slow sampling (hundreds of denoising steps); computationally expensive for large datasets.

---

### 4. Normalizing Flows
**How it works:** A sequence of invertible transformations maps a simple base distribution (e.g., Gaussian) to the data distribution. Exact likelihood computation is possible.  
**Advantage:** Exact latent inference, principled probability estimation, no reconstruction-vs-generation trade-off.  
**Limitation:** Architecture constrained to invertible functions; can be memory-intensive.

---

### 5. Vector Quantized VAE (VQ-VAE)
**How it works:** The latent space is discrete — codes are mapped to the nearest entry in a learned codebook. A second-stage autoregressive or diffusion model is trained over the discrete codes.  
**Advantage:** Very high reconstruction fidelity; discrete latent codes can represent distinct morphology types (blobs, stripes, sparse inclusions) cleanly.  
**Limitation:** Two-stage training required; codebook size must be tuned carefully.

---

### Summary Table

| Method | Latent structure | Generation quality | Training stability | Good for microstructures? |
|--------|-----------------|-------------------|-------------------|---------------------------|
| Standard AE | None (deterministic) | Poor | Stable | Reconstruction only |
| **VAE** | Continuous N(0,I) | Good | Stable | Yes |
| GAN | Implicit | Excellent | Unstable | Yes (sharp images) |
| Diffusion | None (iterative) | State of the art | Stable | Yes (slow) |
| Norm. Flow | Exact density | Good | Moderate | Yes |
| VQ-VAE | Discrete codebook | Very good | Stable | Yes (distinct modes) |

For this problem, a **VAE** is the most practical next step: it adds minimal architectural complexity over the existing autoencoder while providing a well-structured latent space for controlled generation.

---
## References

1. Robertson, A., et al. "An efficient dataset and benchmark for large-scale 2D microstructure generation." *Integrating Materials and Manufacturing Innovation*, 2023. https://doi.org/10.1007/s40192-023-00340-4
2. Kingma, D. P., & Welling, M. (2013). Auto-Encoding Variational Bayes. *arXiv:1312.6114*.
3. Goodfellow, I., et al. (2014). Generative Adversarial Nets. *NeurIPS*.
4. Ho, J., Jain, A., & Abbeel, P. (2020). Denoising Diffusion Probabilistic Models. *NeurIPS*.
5. van den Oord, A., Vinyals, O., & Kavukcuoglu, K. (2017). Neural Discrete Representation Learning (VQ-VAE). *NeurIPS*.
6. Lecture notes: 1.121/2.174/1.052J, Spring 2026, Prof. Markus J. Buehler, MIT.